In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score

def set_seed(seed=10879360):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(10879360)
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

PyTorch: 2.10.0+cu128
GPU: NVIDIA A100-SXM4-40GB


In [3]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/NLU/Group/AV/'

train_df = pd.read_csv(DATA_PATH + 'train.csv')
dev_df = pd.read_csv(DATA_PATH + 'dev.csv')

print(f"Train: {len(train_df):,} | Dev: {len(dev_df):,}")
print(train_df['label'].value_counts())
print(train_df.isnull().sum())

Mounted at /content/drive
Train: 27,643 | Dev: 5,993
label
0.0    13950
1.0    13693
Name: count, dtype: int64
text_1    0
text_2    0
label     0
dtype: int64


In [4]:
!pip install transformers datasets torch -q

In [5]:
MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
BATCH_SIZE = 32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def clean_text(text):
    text = re.sub(r'(From|To|Cc|Subject|Date|Forwarded by)[^\n]*\n', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

class AVDataset(Dataset):
    def __init__(self, df, has_label=True):
        df = df.copy()
        df['text_1'] = df['text_1'].apply(clean_text)
        df['text_2'] = df['text_2'].apply(clean_text)
        self.encodings = tokenizer(
            list(df['text_1']),
            list(df['text_2']),
            max_length=MAX_LENGTH,
            truncation='longest_first',
            padding=False,
            return_tensors=None
        )
        self.labels = df['label'].values if has_label else None
        self.has_label = has_label

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
        }
        if self.has_label:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataset = AVDataset(train_df)
dev_dataset = AVDataset(dev_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2,
                          collate_fn=data_collator)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2,
                        collate_fn=data_collator)

print(f"Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches: 864 | Dev batches: 188


In [6]:
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/NLU/Group/AV/roberta_largel_best'
EPOCHS = 10
LEARNING_RATE = 1e-5

class ASLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos

    def forward(self, logits, labels):
        probs = torch.softmax(logits, dim=-1)
        probs_pos = probs[:, 1]
        probs_neg = probs[:, 0]
        los_pos = labels * torch.log(probs_pos.clamp(min=1e-8))
        los_neg = (1 - labels) * torch.log(probs_neg.clamp(min=1e-8))
        loss = los_pos + los_neg
        loss[labels == 1] *= (1 - probs_pos[labels == 1]) ** self.gamma_pos
        loss[labels == 0] *= (1 - probs_neg[labels == 0]) ** self.gamma_neg
        return -loss.mean()

class AVTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = ASLoss(gamma_neg=4, gamma_pos=1)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,
    seed=10879360,
    save_only_model=True
)

print(f"Training setup complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training setup complete!


In [7]:
import json
from sklearn.metrics import f1_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32
)

trainer = AVTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()
print(f"\nBest Val F1: {trainer.state.best_metric:.4f}")

def find_best_threshold(trainer, dataset):
    predictions = trainer.predict(dataset)
    probs = torch.softmax(
        torch.tensor(predictions.predictions), dim=-1
    )[:, 1].numpy()
    labels = predictions.label_ids

    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.2, 0.8, 0.01):
        preds = (probs >= thresh).astype(int)
        f1 = f1_score(labels, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    print(f"Best threshold: {best_thresh:.2f} | F1: {best_f1:.4f}")
    return best_thresh

best_threshold = find_best_threshold(trainer, dev_dataset)

threshold_data = {'best_threshold': float(best_threshold)}
with open(f'{OUTPUT_DIR}/best_threshold.json', 'w') as f:
    json.dump(threshold_data, f)

print(f"Model saved at: {OUTPUT_DIR}")
print(f"Best threshold saved: {best_threshold:.2f}")

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,0.166232,0.126913,0.667769
2,0.124015,0.108907,0.730109
3,0.099081,0.111519,0.786200
4,0.085935,0.107864,0.793729
5,0.066187,0.115737,0.801308
6,0.052042,0.204728,0.818917
7,0.039252,0.208688,0.826071
8,0.029757,0.266594,0.824795
9,0.023309,0.253853,0.828132
10,0.018228,0.266475,0.827570


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Best Val F1: 0.8281


Best threshold: 0.54 | F1: 0.8295
Model saved at: /content/drive/MyDrive/Colab Notebooks/NLU/Group/AV/roberta_largel_best
Best threshold saved: 0.54
